# Stock Prices Dataset Cleaning and Validation Pipeline

## Overview

This notebook documents the complete cleaning and validation process for the **stock prices dataset** used in the financial analytics ETL pipeline. The objective of this process is to transform raw financial data into a structured analytical dataset that can be safely used for integration, modeling, and dashboard development.

The raw dataset contains historical price data for several technology companies. However, the data is stored in a **wide format structure**, where each company's values appear as separate column groups. This format is not suitable for analytical workflows or database integration.

Therefore, the pipeline performs several transformations to convert the raw dataset into a **tidy analytical format** where each row represents a single observation of a stock on a specific trading day.

## Objectives of the Cleaning Pipeline

The cleaning process was designed to achieve the following goals:

- Transform the dataset from **wide format to long analytical format**
- Ensure that each observation corresponds to **one stock on one trading day**
- Standardize data types across all variables
- Remove non-data rows and structural artifacts from the raw file
- Validate the final dataset for completeness and integrity
- Produce a clean dataset ready for integration with other financial datasets

The resulting dataset will later be integrated with:

- S&P 500 index data
- company metadata

to support advanced financial analysis and visualization.

---

## Raw Dataset Characteristics

The raw dataset initially contained:

- Historical stock prices
- Multiple companies stored in the same table
- separate column groups for each company
- a header row containing ticker symbols

The dataset includes the following companies:

- Apple (AAPL)
- Microsoft (MSFT)
- Amazon (AMZN)
- NVIDIA (NVDA)
- Tesla (TSLA)

Each company includes the following financial metrics:

- Open price
- High price
- Low price
- Close price
- Trading volume

In [8]:
import pandas as pd

# -------------------------------------------------------
# Load raw dataset
# -------------------------------------------------------

file_path = "../../data/raw/stock_prices.csv"
stock_raw = pd.read_csv(file_path)

print("Raw dataset shape:", stock_raw.shape)

# -------------------------------------------------------
# Remove the first row (ticker metadata row)
# -------------------------------------------------------

stock = stock_raw.iloc[1:].reset_index(drop=True)

print("\nDataset after removing ticker row:")
print(stock.head())

# -------------------------------------------------------
# Convert Date column to datetime
# -------------------------------------------------------

stock["Date"] = pd.to_datetime(stock["Date"])

# -------------------------------------------------------
# Convert numeric columns
# -------------------------------------------------------

numeric_columns = stock.columns.drop(["Date","Ticker"], errors="ignore")

stock[numeric_columns] = stock[numeric_columns].apply(
    pd.to_numeric,
    errors="coerce"
)

# -------------------------------------------------------
# Melt dataset (EXCLUDING ticker column)
# -------------------------------------------------------

long_data = pd.melt(
    stock,
    id_vars=["Date"],
    value_vars=[c for c in stock.columns if c not in ["Date","Ticker"]],
    var_name="Variable",
    value_name="Value"
)

print("\nLong format preview:")
print(long_data.head())


Raw dataset shape: (12581, 27)

Dataset after removing ticker row:
         Date               Close                High                 Low  \
0  2015-01-02  24.214893341064453   24.68222562747745  23.776353422468404   
1  2015-01-05      23.53271484375  24.064277681708337  23.346668779850006   
2  2015-01-06  23.534936904907227  23.794073292366384  23.173916188735014   
3  2015-01-07   23.86495018005371  23.964617509529127  23.632390824888006   
4  2015-01-08   24.78188133239746   24.83946773083683  24.075345785125826   

                 Open       Volume Ticker Close.1 High.1 Low.1  ... Close.3  \
0  24.671150729967486  212818400.0   AAPL     NaN    NaN   NaN  ...     NaN   
1  23.984543171525004  257142000.0   AAPL     NaN    NaN   NaN  ...     NaN   
2   23.59695227731193  263188400.0   AAPL     NaN    NaN   NaN  ...     NaN   
3  23.743133053101715  160423600.0   AAPL     NaN    NaN   NaN  ...     NaN   
4  24.192733898426823  237458000.0   AAPL     NaN    NaN   NaN  ...     NaN

# Step 1: Loading the Raw Dataset

The first step in the pipeline is to load the raw dataset from the data storage directory.

The dataset is stored in the **raw data layer** of the project directory. In a typical ETL architecture, the raw layer stores unmodified source data exactly as it was obtained from the original data provider.

At this stage, the dataset is loaded without any transformations so that the original structure can be inspected.

This allows us to observe the following characteristics:

- total number of rows and columns
- column naming structure
- presence of non-data rows
- format of financial variables

Understanding the raw structure is essential before performing any cleaning operations.

# Step 2: Removing the Metadata Row

After loading the dataset, it was observed that the first row of the dataset does not contain actual trading data. Instead, this row contains **ticker symbols** corresponding to each group of financial columns.

For example:

- columns such as `Close`, `High`, `Low`, `Open`, and `Volume` correspond to the stock **AAPL**
- columns such as `Close.1`, `High.1`, `Low.1`, `Open.1`, and `Volume.1` correspond to **MSFT**

This row acts as a secondary header describing which ticker each group of columns belongs to. However, it is incorrectly stored as part of the dataset.

Since this row does not represent an actual trading observation, it must be removed before performing any transformations.

Removing this row ensures that:

- all remaining rows represent valid trading data
- numerical columns can be converted safely
- future reshaping operations behave correctly

# Step 3: Standardizing Data Types

Financial datasets require consistent data types to support analytical operations.

At this stage, the dataset still contains many columns stored as text values due to the presence of non-numeric elements in the raw dataset.

The following conversions are required:

### Date Column
The `Date` column is converted to a **datetime format** so that the dataset can support time-series operations such as:

- chronological sorting
- time-based filtering
- time-series modeling

### Price and Volume Columns
Financial metrics such as:

- Open
- High
- Low
- Close
- Volume

must be converted into **numeric data types**.

This conversion ensures that the values can be used for calculations such as:

- returns
- volatility
- moving averages
- statistical summaries

Any values that cannot be interpreted as numeric values are automatically converted to missing values during this step.

# Step 4: Reshaping the Dataset from Wide Format to Long Format

The raw dataset stores financial data in a **wide format structure**, where each company occupies its own group of columns.

For example:

| Date | Close | Close.1 | Close.2 | ... |
|-----|------|------|------|------|

In this format:

- Close → AAPL
- Close.1 → MSFT
- Close.2 → AMZN

While this format may be convenient for spreadsheet viewing, it is unsuitable for analytical workflows.

Most analytical systems require data in **tidy format**, where:

- each row represents one observation
- each column represents one variable

Therefore, the dataset is reshaped using a **melt transformation**, which converts multiple column groups into a single column containing the metric values.

After melting, the dataset contains three columns:

- Date
- Variable (metric name)
- Value (metric value)

This structure allows us to extract the metric type and the corresponding ticker symbol.

In [2]:
# -------------------------------------------------------
# Extract metric and ticker group
# -------------------------------------------------------

long_data["Metric"] = long_data["Variable"].str.split(".").str[0]
long_data["Group"] = long_data["Variable"].str.split(".").str[1]


In [3]:

# -------------------------------------------------------
# Map ticker groups to ticker names
# -------------------------------------------------------

ticker_map = {
    None: "AAPL",
    "1": "MSFT",
    "2": "AMZN",
    "3": "NVDA",
    "4": "TSLA"
}

long_data["Ticker"] = long_data["Group"].map(ticker_map)
long_data["Ticker"] = long_data["Ticker"].fillna("AAPL")


print("\nTicker assignment preview:")
print(long_data[["Variable","Metric","Group","Ticker"]].head())




Ticker assignment preview:
  Variable Metric Group Ticker
0    Close  Close   NaN   AAPL
1    Close  Close   NaN   AAPL
2    Close  Close   NaN   AAPL
3    Close  Close   NaN   AAPL
4    Close  Close   NaN   AAPL


# Step 5: Extracting Financial Metrics and Ticker Identifiers

After reshaping the dataset, each row contains a variable name that includes both the financial metric and a group identifier.

Examples of these variable names include:

- Close
- Close.1
- Close.2
- Close.3

The portion before the period represents the **financial metric**, while the number after the period represents the **ticker group identifier**.

For example:

- Close → metric for the first company
- Close.1 → metric for the second company

This step separates the variable column into two components:

1. Metric name (Open, High, Low, Close, Volume)
2. Group identifier corresponding to a specific ticker

# Step 6: Mapping Ticker Groups to Company Symbols

Once the metric and group identifiers are extracted, the next step is to associate each group with its corresponding ticker symbol.

Based on the structure of the dataset, the following mapping was identified:

| Group | Ticker |
|------|------|
| None | AAPL |
| 1 | MSFT |
| 2 | AMZN |
| 3 | NVDA |
| 4 | TSLA |

This mapping allows us to assign a ticker symbol to every observation in the dataset.

After this step, every row contains a clear identifier for the company whose trading data is being recorded.

# Step 7: Reconstructing the Analytical Dataset

After the ticker symbols and financial metrics are identified, the dataset must be reorganized into its final analytical structure.

The melt transformation created a dataset where each row contains a single metric value. To make the dataset suitable for analysis, the metrics are pivoted back into columns.

The final dataset contains the following columns:

- Date
- Ticker
- Open
- High
- Low
- Close
- Volume

Each row now represents:

**one stock on one trading day**

This structure is widely used in financial data analysis and is compatible with:

- relational databases
- machine learning models
- business intelligence dashboards

In [4]:


# -------------------------------------------------------
# Pivot dataset into analytical structure
# -------------------------------------------------------

stock_final = long_data.pivot_table(
    index=["Date","Ticker"],
    columns="Metric",
    values="Value"
).reset_index()


print("\nFinal cleaned dataset:")
print(stock_final.head())


Final cleaned dataset:
Metric       Date Ticker      Close       High        Low       Open  \
0      2015-01-02   AAPL  24.214893  24.682226  23.776353  24.671151   
1      2015-01-02   AMZN  15.426000  15.737500  15.348000  15.629000   
2      2015-01-02   MSFT  39.767685  40.328991  39.580585  39.682640   
3      2015-01-02   NVDA   0.483011   0.486611   0.475333   0.483011   
4      2015-01-02   TSLA  14.620667  14.883333  14.217333  14.858000   

Metric       Volume  
0       212818400.0  
1        55664000.0  
2        27913900.0  
3       113680000.0  
4        71466000.0  


In [5]:

print("\n====================================================")
print("POST-CLEANING DATA VALIDATION REPORT")
print("====================================================")


# ----------------------------------------------------
# Dataset Shape
# ----------------------------------------------------

print("\n1. DATASET SHAPE")
print(stock_final.shape)


# ----------------------------------------------------
# Column Structure
# ----------------------------------------------------

print("\n2. COLUMN NAMES")
print(stock_final.columns)


# ----------------------------------------------------
# Data Types
# ----------------------------------------------------

print("\n3. DATA TYPES")
print(stock_final.dtypes)


# ----------------------------------------------------
# Missing Values
# ----------------------------------------------------

print("\n4. MISSING VALUES")
print(stock_final.isnull().sum())


# ----------------------------------------------------
# Duplicate Rows
# ----------------------------------------------------

print("\n5. DUPLICATE ROWS")
duplicates = stock_final.duplicated().sum()
print("Duplicate records:", duplicates)


# ----------------------------------------------------
# Duplicate Date + Ticker combinations
# ----------------------------------------------------

print("\n6. DUPLICATE DATE-TICKER COMBINATIONS")

dup_keys = stock_final.duplicated(subset=["Date","Ticker"]).sum()
print("Duplicate Date + Ticker:", dup_keys)


# ----------------------------------------------------
# Unique Tickers
# ----------------------------------------------------

print("\n7. UNIQUE TICKERS")
print(stock_final["Ticker"].unique())


# ----------------------------------------------------
# Date Range
# ----------------------------------------------------

print("\n8. DATE RANGE")

print("Minimum date:", stock_final["Date"].min())
print("Maximum date:", stock_final["Date"].max())


# ----------------------------------------------------
# Numeric Summary
# ----------------------------------------------------

print("\n9. NUMERIC SUMMARY")
print(stock_final.describe())


# ----------------------------------------------------
# Negative Values Check
# ----------------------------------------------------

print("\n10. NEGATIVE VALUE CHECK")

numeric_cols = ["Open","High","Low","Close","Volume"]

for col in numeric_cols:
    negatives = (stock_final[col] < 0).sum()
    print(f"{col}: {negatives} negative values")


# ----------------------------------------------------
# Logical Price Check
# ----------------------------------------------------

print("\n11. LOGICAL PRICE CHECK")

invalid_prices = stock_final[
    (stock_final["High"] < stock_final["Low"])
]

print("Rows where High < Low:", len(invalid_prices))


# ----------------------------------------------------
# Sample Records
# ----------------------------------------------------

print("\n12. SAMPLE RECORDS")
print(stock_final.head(10))


print("\n====================================================")
print("VALIDATION COMPLETE")
print("====================================================")


POST-CLEANING DATA VALIDATION REPORT

1. DATASET SHAPE
(12580, 7)

2. COLUMN NAMES
Index(['Date', 'Ticker', 'Close', 'High', 'Low', 'Open', 'Volume'], dtype='object', name='Metric')

3. DATA TYPES
Metric
Date      datetime64[ns]
Ticker            object
Close            float64
High             float64
Low              float64
Open             float64
Volume           float64
dtype: object

4. MISSING VALUES
Metric
Date      0
Ticker    0
Close     0
High      0
Low       0
Open      0
Volume    0
dtype: int64

5. DUPLICATE ROWS
Duplicate records: 0

6. DUPLICATE DATE-TICKER COMBINATIONS
Duplicate Date + Ticker: 0

7. UNIQUE TICKERS
['AAPL' 'AMZN' 'MSFT' 'NVDA' 'TSLA']

8. DATE RANGE
Minimum date: 2015-01-02 00:00:00
Maximum date: 2024-12-31 00:00:00

9. NUMERIC SUMMARY
Metric                           Date         Close          High  \
count                           12580  12580.000000  12580.000000   
mean    2019-12-31 19:03:31.764705792    102.346291    103.686998   
min        

# Step 8: Post-Cleaning Data Validation

Before saving the cleaned dataset, a comprehensive validation step is performed to ensure that the transformation process did not introduce errors.

The validation process checks the following aspects of the dataset:

### Dataset Shape
Verifies the number of rows and columns to ensure that the transformation produced the expected number of observations.

### Column Structure
Ensures that the dataset contains the correct variables required for financial analysis.

### Missing Values
Confirms that no essential values are missing after transformation.

### Duplicate Records
Checks for duplicate rows that could distort analytical results.

### Duplicate Date-Ticker Pairs
Ensures that each stock has only one record per trading day.

### Date Range
Verifies the historical coverage of the dataset.

### Numeric Summary
Examines descriptive statistics to detect abnormal values.

### Logical Price Rules
Ensures that the following financial rule holds for every observation:


In [6]:

# -------------------------------------------------------
# Save cleaned dataset
# -------------------------------------------------------

output_path = "../../data/cleaned/stock_prices_clean.csv"

stock_final.to_csv(output_path, index=False)

print("\nClean dataset saved to clean folder.")


Clean dataset saved to clean folder.


# Step 9: Saving the Clean Dataset

After completing the validation checks and confirming that the dataset is free of structural issues, the cleaned dataset is saved to the **clean data layer** of the project.

The clean layer stores datasets that have been processed and validated but are not yet integrated with other data sources.

The cleaned dataset will later be combined with additional financial datasets, including:

- S&P 500 market index data
- company metadata

This integration will allow the creation of a comprehensive dataset for financial analysis and dashboard development.